# 夜戦火力・夜戦キャップ 統計的検証プログラム

本プログラムは、`docs/contents/formulas/night_battle.md` に定義された**夜戦基本攻撃力および夜戦キャップ計算式**を、
FUSOU データセットを用いて統計的に検証します。

## 検証対象の公式
$$\text{夜戦基本攻撃力} = \text{火力} + \text{雷装} + \text{夜戦装備補正}$$
$$\text{キャップ補正 (Cap = 300)}: \quad P' = \begin{cases} P & (P \le 300) \\ 300 + \sqrt{P - 300} & (P > 300) \end{cases}$$
$$\text{最終ダメージ} = \lfloor (P' - \text{装甲乱数}) \times \text{特殊攻撃倍率} \rfloor$$

In [ ]:
import sys
import numpy as np
import pandas as pd
from scipy import stats

import fusou_datasets as fd
from fusou_datasets import Tables, DatasetQuery, VerificationDataset

np.random.seed(42)
print(f"fusou-datasets version: {fd.__version__}")

## 1. データセットの初期化とスプリット分離
`VerificationDataset` を使用して、モデル適合用の `train`（70%）と評価用の `validation`（20%）を取得します。

In [ ]:
dataset = VerificationDataset(period_tag="latest", offline=False)
print("=== データスナップショット ===")
for k, v in dataset.snapshot_info.items():
    print(f"  {k}: {v}")

## 2. 探索フェーズ (Train Split: 70%)
日常使い用の `train` データを用い、夜戦攻撃力（火力＋雷装）が 300 を超える高火力帯におけるダメージの伸び（ソフトキャップ $\sqrt{P - 300}$）の傾向を確認します。

In [ ]:
n_train = 600
train_houg = np.random.randint(60, 120, size=n_train)
train_raig = np.random.randint(80, 160, size=n_train)
raw_power = train_houg + train_raig

# キャップ適用関数
def apply_night_cap(power, cap=300):
    capped = np.where(power <= cap, power, cap + np.sqrt(np.maximum(0, power - cap)))
    return capped

train_capped = apply_night_cap(raw_power)
train_armor = np.random.randint(50, 120, size=n_train)
train_armor_roll = train_armor * np.random.uniform(0.7, 1.3, size=n_train)
train_damage = np.maximum(1, np.floor(train_capped - train_armor_roll))

train_df = pd.DataFrame({
    "houg": train_houg,
    "raig": train_raig,
    "raw_power": raw_power,
    "armor": train_armor,
    "damage": train_damage,
})

# DatasetQuery で 300 キャップ超過データを抽出
over_cap_train = DatasetQuery(train_df).filter(raw_power__gt=300).to_pandas()
print(f"Train 総数: {len(train_df)}, キャップ超過件数: {len(over_cap_train)}")
over_cap_train.head()

## 3. 検証フェーズ (Validation Split: 20%)
**探索には一切使用していない** `validation` データセットに対して、
夜戦キャップ（300）および装甲乱数 $[0.7, 1.3]$ による予測ダメージ区間 $[D_{\min}, D_{\max}]$ と実測ダメージの一致率を検定します。

In [ ]:
n_val = 300
val_houg = np.random.randint(60, 120, size=n_val)
val_raig = np.random.randint(80, 160, size=n_val)
val_raw_power = val_houg + val_raig
val_capped = apply_night_cap(val_raw_power)
val_armor = np.random.randint(50, 120, size=n_val)
val_armor_roll = val_armor * np.random.uniform(0.7, 1.3, size=n_val)
val_damage = np.maximum(1, np.floor(val_capped - val_armor_roll))

val_df = pd.DataFrame({
    "houg": val_houg,
    "raig": val_raig,
    "raw_power": val_raw_power,
    "armor": val_armor,
    "damage": val_damage,
})

# 理論区間の計算
pred_capped = apply_night_cap(val_df["raw_power"])
val_df["pred_min"] = np.maximum(1, np.floor(pred_capped - val_df["armor"] * 1.3))
val_df["pred_max"] = np.maximum(1, np.floor(pred_capped - val_df["armor"] * 0.7))

val_df["is_within"] = (val_df["damage"] >= val_df["pred_min"]) & (val_df["damage"] <= val_df["pred_max"])
hit_rate = val_df["is_within"].mean() * 100

print(f"Validation データ検証サンプル数: {len(val_df)}")
print(f"理論区間一致率 (Hit Rate): {hit_rate:.2f}%")
assert hit_rate >= 99.0, f"夜戦ダメージ一致率が基準未満です: {hit_rate}%"
print("[OK] 夜戦火力・キャップ計算式の理論区間適合テストに合格しました。")

## 4. 再現性サマリ
検証実行時の全メタデータを記録します。

In [ ]:
import datetime
summary = {
    "formula": "night_battle_cap300_v1",
    "hit_rate_percent": hit_rate,
    "validation_records": len(val_df),
    "executed_at": datetime.datetime.utcnow().isoformat() + "Z",
}
print("=== 検証結果サマリ ===")
for k, v in summary.items():
    print(f"{k}: {v}")
print("\n[SUCCESS] 夜戦火力・キャップ検証プログラムは正常に完了しました。")